# 03 · Retrieval — Query 技巧（Rewrite / Multi-Query / HyDE）

目标：
- 复用课件里的“Query 改写”思路
- 在同一个 Chroma collection 上对比：
  - baseline（原始 query）
  - query rewrite（LLM 改写）
  - multi-query（生成多条查询，融合召回）
  - HyDE（先生成假设答案，再用它去检索）

> 依赖：上一节已写入 `data/chroma` + 设置 `OPENAI_API_KEY`。


In [4]:
pip install langchain_openai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.2-py3-none-any.whl (1.0 MB)
Using cached la

In [18]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
embed_model =  os.getenv("EMBED_MODEL") # for Openrouter "qwen/qwen3-embedding-4b"
chat_model = os.getenv("CHAT_MODEL") 

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

SECTION_COLLECTION = "autel_annual_report_2024_sections"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}
print(embed_model)
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)

import chromadb
_chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

vs = Chroma(collection_name=COLLECTION, embedding_function=emb, client=_chroma_client)
section_vs = Chroma(collection_name=SECTION_COLLECTION, embedding_function=emb, client=_chroma_client)

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
print("ready — chunk collection:", COLLECTION, "| section collection:", SECTION_COLLECTION)
print("embed:", embed_model, "| chat:", chat_model, "| env:", ENV_FILE)


Qwen/Qwen3-Embedding-8B
ready — chunk collection: autel_annual_report_2024 | section collection: autel_annual_report_2024_sections
embed: Qwen/Qwen3-Embedding-8B | chat: deepseek-ai/DeepSeek-V3.2 | env: /Users/mengbai/Documents/AI-training/.env


### Baseline

In [ ]:
# baseline 检索


QUESTION = "道通公司的AI战略是？"

hits = vs.similarity_search(QUESTION, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content.replace("\n", " "))
    print()


[1] {'file_path': 'mineru/autel_annual_report_2024.md', 'chunk_in_section': 0, 'doc_group': 'full_document', 'chunk_id': 136, 'source_doc_count': 1, 'section_title': '五、报告期内主要经营情况', 'doc_id': '道通24年年报__full', 'content_level': 'chunk', 'chunk_in_doc': 136, 'section_in_doc': 119, 'section_id': '道通24年年报__full::section_119', 'h1': '五、报告期内主要经营情况', 'parse_source': 'mineru', 'source': '道通24年年报'}
报告期内，公司实现营业收入 393,225.64万元，同比增长 $2 0 . 9 5 \%$ ；归属于上市公司股东净利 64,092.52万元，同比增长 $2 5 7 . 5 9 \%$ ；归属于上市公司股东的扣除非经常性损益净利润54,077.44万元，同比增加 $4 7 . 4 2 \%$ 。

[2] {'section_id': '道通24年年报__full::section_131', 'content_level': 'chunk', 'doc_id': '道通24年年报__full', 'chunk_id': 155, 'source': '道通24年年报', 'h1': 'B.公司主要供应商情况', 'chunk_in_section': 0, 'file_path': 'mineru/autel_annual_report_2024.md', 'section_title': 'B.公司主要供应商情况', 'parse_source': 'mineru', 'section_in_doc': 131, 'source_doc_count': 1, 'chunk_in_doc': 155, 'doc_group': 'full_document'}
$\surd$ 适用 □不适用   前五名供应商采购额33,582.16万元，占年度采购总额 $1 7 . 7 3 \%$ ；其中前五

In [20]:
### Query Rewrite

In [28]:
# 1) Query Rewrite：根据问题主题选择更贴近年报原文标题的检索锚点

from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Query Rewrite 模块。"
            "请先在心里识别用户问题主题，再输出 1 条更适合向量检索的中文 query。\n"
            "原则：\n"
            "1. 保留公司名、年份和核心问题，不要扩大或改变问题范围；\n"
            "2. 优先使用年报中可能真实出现的章节名、小节名、关键词锚点，而不是泛泛改写；\n"
            "3. 如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先考虑这类锚点：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势；\n"
            "4. 如果问题是其他主题，也按同样思路改写成更贴近年报标题的 query；\n"
            "5. 只输出 1 行 query，不要解释，不要回答问题。",
        ),
        ("human", "原始问题：{q}"),
    ]
)

rewritten = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
print("rewritten:\n", rewritten)

hits = vs.similarity_search(rewritten, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


rewritten:
 AI战略
[1] {'source_doc_count': 1, 'chunk_in_doc': 850, 'doc_group': 'full_document', 'doc_id': '道通24年年报__full', 'content_level': 'chunk', 'h1': '(2). 应付项目', 'source': '道通24年年报', 'section_title': '(2). 应付项目', 'file_path': 'mineru/autel_annual_report_2024.md', 'chunk_id': 850, 'section_id': '道通24年年报__full::section_733', 'parse_source': 'mineru', 'section_in_doc': 733, 'chunk_in_section': 0}
√适用 □不适用   单位：元币种：人民币   | 项目名称 | 关联方 | 期末账面余额 | 期初账面余额 | | --- | --- | --- | --- | | 应付账款 | 智能航空 | 22,558.99 | 198,563.30 |

[2] {'doc_group': 'full_document', 'chunk_in_doc': 762, 'source': '道通24年年报', 'section_in_doc': 654, 'section_id': '道通24年年报__full::section_654', 'section_title': '(5). 使用范围受限但仍作为现金和现金等价物列示的情况', 'chunk_id': 762, 'parse_source': 'mineru', 'content_level': 'chunk', 'file_path': 'mineru/autel_annual_report_2024.md', 'source_doc_count': 1, 'chunk_in_section': 0, 'h1': '(5). 使用范围受限但仍作为现金和现金等价物列示的情况', 'doc_id': '道通24年年报__full'}
√适用 □不适用   单位：元币种：人民币   | 项目 | 本期金额 | 理由 | | ---

### Multi-Query

In [29]:
# 2) Multi-Query：围绕同一问题生成“不同标题锚点”的多路检索 query

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Multi-Query 生成器。"
            "给定一个问题，请先识别问题主题，再输出 4 条中文 query，每条一行，不要编号，不要解释。\n"
            "这 4 条 query 必须分别覆盖以下 4 种角度：\n"
            "1. 原问题压缩版：保留公司、年份、核心主题；\n"
            "2. 章节标题版：优先使用年报里可能真实出现的章节/小节标题；\n"
            "3. 子主题展开版：把该主题拆成 2 到 4 个最可能回答问题的子点；\n"
            "4. 关键词聚合版：把公司、年份、主题词、近义词和标题锚点组合成一个更像检索式的 query；\n"
            "如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先围绕这些词生成：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势。\n"
            "要求：\n"
            "- 保留公司名和年份；\n"
            "- 4 条 query 必须明显不同，不能只是换同义词；\n"
            "- 不要编造数字；\n"
            "- 只输出 4 行 query。",
        ),
        ("human", "问题：{q}"),
    ]
)

queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
print("queries:")
for q in queries:
    print("-", q)

# 融合策略：RRF（Reciprocal Rank Fusion）
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}

for q in queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)

merged = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)]

print("merged hits:", len(merged))
for i, d in enumerate(merged[:8], 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


queries:
- 报告期内公司AI战略
- 公司AI战略规划与实施
- 公司AI战略：技术研发、产品应用、生态合作
- 公司 AI战略 人工智能 规划 布局 发展
merged hits: 30
[1] {'doc_group': 'full_document', 'parse_source': 'mineru', 'section_in_doc': 39, 'source_doc_count': 1, 'content_level': 'chunk', 'chunk_in_doc': 43, 'chunk_in_section': 0, 'section_title': '十二、因国家秘密、商业秘密等原因的信息暂缓、豁免情况说明', 'section_id': '道通24年年报__full::section_39', 'chunk_id': 43, 'file_path': 'mineru/autel_annual_report_2024.md', 'source': '道通24年年报', 'h1': '十二、因国家秘密、商业秘密等原因的信息暂缓、豁免情况说明', 'doc_id': '道通24年年报__full'}
$\surd$ 适用 □不适用   因公司与部分客户、供应商的合作信息涉及商业秘密，根据《上海证券交易所科创板股票上市规则》《上海证券交易所科创板上市公司自律监管指引第 1 号——规范运作》的相关规定，公司已按照《信息披露管理制度》完成相应的信息披露豁免审批程序。

[2] {'file_path': 'mineru/autel_annual_report_2024.md', 'chunk_in_doc': 323, 'source_doc_count': 1, 'content_level': 'chunk', 'chunk_id': 323, 'section_title': '1、已在临时公告披露且后续实施无进展或变化的事项', 'section_in_doc': 273, 'doc_group': 'full_document', 'section_id': '道通24年年报__full::section_273', 'doc_id': '道通24年年报__full', 'parse_source': 'mineru', 'source'

### HyDE

In [30]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。请为用户问题写一段可能出现在年报中的‘假设答案’，用正式书面语，尽量包含可检索的关键词（业务、产品线、收入、分部等）。不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)

hypo = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
print("hypo (first 400 chars):\n", hypo[:400])

hits = vs.similarity_search(hypo, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()

hypo (first 400 chars):
 本集团将人工智能技术作为核心战略驱动力，持续加大研发投入，致力于将AI能力深度融入各主要产品线及业务运营流程。在智能云服务、企业解决方案及消费级硬件等关键分部，我们通过自研大模型与机器学习平台，显著提升了产品智能化水平与服务效率。AI相关技术不仅优化了现有业务的用户体验与运营效率，亦在自动驾驶、智能医疗等新兴领域孵化了具有潜力的增长点。公司已成立专项AI伦理委员会，确保技术发展符合负责任创新原则。未来，我们将继续深化“AI赋能”战略，构建开放的技术生态，以巩固长期竞争优势并驱动可持续增长。
[1] {'section_id': '道通24年年报__full::section_92', 'chunk_in_doc': 109, 'section_in_doc': 92, 'content_level': 'chunk', 'chunk_in_section': 0, 'file_path': 'mineru/autel_annual_report_2024.md', 'source': '道通24年年报', 'doc_group': 'full_document', 'doc_id': '道通24年年报__full', 'h1': '1、技术创新优势', 'chunk_id': 109, 'parse_source': 'mineru', 'section_title': '1、技术创新优势', 'source_doc_count': 1}
公司长期以来高度重视研发创新，将其作为保持市场竞争力的核心动力。公司以AI技术为核心驱动引擎，不断投入高强度研发资源，确保技术创新与市场需求的精准对接，2024年公司研发投入金额为6.80亿元，占营业收入的比例为 $1 7 . 2 9 \%$ 。    在数字维修业务领域，公司充分发挥汽车协议及实车测试资源的优势，通过长期的积累和优化，构建了庞大而丰富的车辆协议信息数据库和核心算法库。这一资源优势不仅使公司具备“超强兼容性”、“车型覆盖面广”和“智能精准”等显著优势，更为生成式 AI 在数字维修领域的行业应用奠定了

[2] {'source_doc_count': 1, 'chunk_id': 56, 'parse_source': 'mineru', 'file_path': 

### Parent-Child

**核心思想：** 检索需要细粒度（chunk 级别），但 LLM 生成需要大背景（section 级别）。

做法：
1. 用 **chunk-level** collection 做向量检索，命中最相关的 chunk
2. 从命中的 chunk metadata 里拿到 `section_id`
3. 用 `section_id` 去 **section-level** collection 查回完整的 section 文本
4. 把 section 文本交给 LLM，而不是只给小 chunk

这解决了 RAG 里经典的"检索精度 vs. 上下文完整性"矛盾。

In [31]:
# 4) Small-to-Big：在 chunk 上检索，返回 parent section 给 LLM

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def small_to_big_retrieve(query: str, k: int = 5, verbose: bool = True):
    """在 chunk collection 上检索，再通过 section_id 回溯 parent section。"""
    chunk_hits = vs.similarity_search(query, k=k)

    section_ids_seen = set()
    parent_sections = []

    for hit in chunk_hits:
        sid = hit.metadata.get("section_id")
        if not sid or sid in section_ids_seen:
            continue
        section_ids_seen.add(sid)

        section_result = section_vs.get(ids=[sid])
        if section_result and section_result["documents"]:
            parent_sections.append({
                "section_id": sid,
                "section_title": section_result["metadatas"][0].get("section_title", ""),
                "text": section_result["documents"][0],
                "triggered_by_chunks": [
                    h.metadata.get("chunk_in_section")
                    for h in chunk_hits
                    if h.metadata.get("section_id") == sid
                ],
            })

    if verbose:
        print(f"query: {query}")
        print(f"chunk hits: {len(chunk_hits)} → unique parent sections: {len(parent_sections)}\n")
        for i, sec in enumerate(parent_sections, 1):
            print(f"[section {i}] {sec['section_id']}")
            print(f"  title: {sec['section_title']}")
            print(f"  triggered by chunk_in_section: {sec['triggered_by_chunks']}")
            print(f"  text preview: {sec['text'][:300].replace(chr(10), ' ')}")
            print()

    return parent_sections


parent_sections = small_to_big_retrieve(QUESTION, k=5)


query: 公司的AI战略是？
chunk hits: 5 → unique parent sections: 5

[section 1] 道通24年年报__full::section_119
  title: 五、报告期内主要经营情况
  triggered by chunk_in_section: [0]
  text preview: 报告期内，公司实现营业收入 393,225.64万元，同比增长 $2 0 . 9 5 \%$ ；归属于上市公司股东净利 64,092.52万元，同比增长 $2 5 7 . 5 9 \%$ ；归属于上市公司股东的扣除非经常性损益净利润54,077.44万元，同比增加 $4 7 . 4 2 \%$ 。

[section 2] 道通24年年报__full::section_131
  title: B.公司主要供应商情况
  triggered by chunk_in_section: [0]
  text preview: $\surd$ 适用 □不适用   前五名供应商采购额33,582.16万元，占年度采购总额 $1 7 . 7 3 \%$ ；其中前五名供应商采购额中关联方采购额0万元，占年度采购总额 $0 \%$ 。

[section 3] 道通24年年报__full::section_128
  title: A.公司主要销售客户情况
  triggered by chunk_in_section: [0]
  text preview: $\surd$ 适用 □不适用   前五名客户销售额98,127.13万元，占年度销售总额 $2 4 . 9 5 \%$ ；其中前五名客户销售额中关联方销售额0万元，占年度销售总额 $0 \%$ 。

[section 4] 道通24年年报__full::section_558
  title: (1). 无形资产情况
  triggered by chunk_in_section: [2]
  text preview: √适用 □不适用   单位：元币种：人民币   | 项目 | 土地使用权 | 专利权 | 商标 | 专有技术 | 软件 | 合计 | | --- | --- | --- | --- | --- | --- | --- | | 一.账面原值 | | 1.期初余额 

### 同题回答对比：Baseline / Rewrite / Multi-Query(RRF) / HyDE / Parent-Child

In [ ]:
import pandas as pd
from IPython.display import display
from langchain_core.prompts import ChatPromptTemplate


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是年报分析助手。基于给定检索内容回答问题。"
            "优先给出结构化结论；若证据不足，明确说明缺失信息。\n\n"
            "检索内容：\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def get_doc_title(d):
    return d.metadata.get("section_title") or d.metadata.get("h3") or d.metadata.get("h2") or "无标题"


def build_context_from_docs(docs, max_docs=5, max_chars=1200):
    blocks = []
    for d in docs[:max_docs]:
        blocks.append(f"【{get_doc_title(d)}】\n{d.page_content[:max_chars]}")
    return "\n\n---\n\n".join(blocks)


def build_context_from_sections(sections, max_sections=5, max_chars=2000):
    return "\n\n---\n\n".join(
        f"【{sec['section_title']}】\n{sec['text'][:max_chars]}"
        for sec in sections[:max_sections]
    )


def answer_with_context(context):
    return llm.invoke(
        answer_prompt.format_messages(context=context, question=QUESTION)
    ).content.strip()


# 1) baseline
baseline_docs = vs.similarity_search(QUESTION, k=5)

# 2) rewrite
rewritten_q = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
rewrite_docs = vs.similarity_search(rewritten_q, k=5)

# 3) multi-query + RRF
multi_queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}
for q in multi_queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
multi_docs = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)][:5]

# 4) HyDE
hypo_q = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
hyde_docs = vs.similarity_search(hypo_q, k=5)

# 5) Parent-Child (Small-to-Big)
parent_sections = small_to_big_retrieve(QUESTION, k=5, verbose=False)
parent_context = build_context_from_sections(parent_sections)

records = [
    {
        "strategy": "Baseline",
        "retrieval_query": QUESTION,
        "hits": len(baseline_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in baseline_docs),
        "answer": answer_with_context(build_context_from_docs(baseline_docs)),
    },
    {
        "strategy": "Rewrite",
        "retrieval_query": rewritten_q,
        "hits": len(rewrite_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in rewrite_docs),
        "answer": answer_with_context(build_context_from_docs(rewrite_docs)),
    },
    {
        "strategy": "Multi-Query (RRF)",
        "retrieval_query": " | ".join(multi_queries),
        "hits": len(multi_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in multi_docs),
        "answer": answer_with_context(build_context_from_docs(multi_docs)),
    },
    {
        "strategy": "HyDE",
        "retrieval_query": hypo_q[:2000].replace("\n", " ") + ("..." if len(hypo_q) > 2000 else ""),
        "hits": len(hyde_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in hyde_docs),
        "answer": answer_with_context(build_context_from_docs(hyde_docs)),
    },
    {
        "strategy": "Parent-Child",
        "retrieval_query": QUESTION,
        "hits": len(parent_sections),
        "hit_titles": " | ".join(sec["section_title"] for sec in parent_sections),
        "answer": answer_with_context(parent_context),
    },
]

comparison_df = pd.DataFrame(records)
comparison_df.insert(0, "question", QUESTION)

pd.set_option("display.max_colwidth", 180)
display(comparison_df)
